# 01 · JET flags and entry-level meta table
Step 1 of the thesis workflow.

**What this notebook does**
1. Loads `journal_entries_4203.csv` (posting lines, `;`-separated) and reduces it to one row per journal entry (`posting_id`).
2. Builds the four JET flags from the **dataset's rule attributes** (main setting, comparable with Gronewald, Scherer, et al. 2026).
3. Re-implements the JETs from **raw fields** and compares them (validation only).
4. Saves the entry-level meta table `data/derived/meta.parquet`.

The meta table is **not** a model input. It is used for ordering (JET-only baseline), filtering (prefilter, H1),
the combined score, H2 flags, the FY split and evaluation.

**Expected folder layout** (the notebook finds the project root itself, so it can run from `notebooks/` or on Colab):
```
<project root>/
├── data/raw/journal_entries_4203.csv   # raw file, never modified
└── data/derived/                       # created by this notebook
```

In [1]:
# cell_01_01
import sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)

RAW_REL = Path("data/raw/journal_entries_4203.csv")


def find_project_root(marker: Path = RAW_REL) -> Path:
    """Walk up from the working directory until `marker` exists."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"{marker} not found from {here} upwards")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))          # makes src/ importable (also on Colab)
from src.load import load_lines, build_entries
from src.jets import apply_jets, JET_COLS, TOP_N, HIGH_CASH_EUR, DELAY_DAYS

DATA = ROOT / RAW_REL
OUT = ROOT / "data" / "derived"
OUT.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

project root: C:\MyWorks\Thesis\Jet


## 0. Constants
Expected sizes come from the dataset paper (Data in Brief 69, 2026, 113257). The JET thresholds of the raw
re-implementation are defined once in `src/jets.py` (J1 top 1,000 amounts per FY, J2 cash credit > EUR 1,500,
J3 lag ≥ 10 days, J4 weekend or outside 08:00–17:59, working hours from `src/load.py`).

In [2]:
# cell_01_02
# expected dataset sizes
N_LINES, N_ENTRIES, N_ANOMALOUS_ENTRIES = 256_232, 102_429, 850

JETS = JET_COLS                     # ["J1", "J2", "J3", "J4"], from src/jets.py
print(f"TOP_N={TOP_N}, HIGH_CASH_EUR={HIGH_CASH_EUR}, DELAY_DAYS={DELAY_DAYS}")

TOP_N=1000, HIGH_CASH_EUR=1500, DELAY_DAYS=10


## 1. Load posting lines

In [3]:
# cell_01_03
lines = load_lines(DATA)            # src/load.py: ';'-separated, dates parsed, adds fy and line_no
assert len(lines) == N_LINES, lines.shape
assert lines["posting_id"].nunique() == N_ENTRIES
lines.head(3)

,posting_id,document_date,posting_date,entered_date,gl_account,gl_account_name,amount,cd_flag,user,tax_rate,...,text,promptly,weekend,nwh,label,marking,top_n,high_cash,fy,line_no
0,a_1,2023-05-31,2023-05-31,2023-06-01 11:52:40,3400,"Cost of merchandise, 19% input tax",112.42,D,David,0.19,...,Rake,1,0,0,0,0,0,0,2023,0
1,a_1,2023-05-31,2023-05-31,2023-06-01 11:52:40,1576,"Deductible input tax, 19%",21.36,D,David,0.19,...,Rake,1,0,0,0,0,0,0,2023,1
2,a_1,2023-05-31,2023-05-31,2023-06-01 11:52:40,1600,Trade payables,133.78,C,David,0.19,...,Rake,1,0,0,0,0,0,0,2023,2


## 2. One row per journal entry
All entry-level attributes (dates, rule attributes, label, marking) are identical on every line of an entry,
so we check that once and then take the first line instead of aggregating each column.
`amount_max` (largest line amount) is the entry amount used for the raw J1 check.

In [4]:
# cell_01_04
ENTRY_COLS = ["document_date", "posting_date", "entered_date",
              "promptly", "weekend", "nwh", "top_n", "high_cash", "label", "marking"]

g = lines.groupby("posting_id", sort=False)          # sort=False: keep CSV order
varies = g[ENTRY_COLS].nunique().gt(1).sum()
assert varies.sum() == 0, f"entry-level columns vary within an entry:\n{varies[varies > 0]}"

entries = g[ENTRY_COLS].first()
entries["n_lines"] = g.size()
entries["amount_max"] = g["amount"].max()
assert (entries["document_date"].dt.year == entries["posting_date"].dt.year).all()
entries.shape

(102429, 12)

## 3. Entry-level meta table (main setting: dataset rule attributes)
| column | definition |
|---|---|
| `fy` | year of `posting_date` |
| `n_lines` | number of posting lines |
| `label`, `marking` | entry label and anomaly type (evaluation only) |
| `J1` | `top_n` (top 1,000 amounts per FY, dataset tie-break) |
| `J2` | `high_cash` (cash postings > EUR 1,500) |
| `J3` | `promptly >= 2` (entry lag ≥ 10 days) |
| `J4` | `weekend > 0` or `nwh == 1` |
| `h` | number of JETs hit (0–4) |
| `jet_any` | `h > 0` (member of the JET population) |

In [5]:
# cell_01_05
meta = pd.DataFrame({
    "fy": entries["posting_date"].dt.year.astype("int16"),
    "n_lines": entries["n_lines"],
    "label": entries["label"],
    "marking": entries["marking"],
    "J1": entries["top_n"],
    "J2": entries["high_cash"],
    "J3": entries["promptly"].ge(2),
    "J4": entries["weekend"].gt(0) | entries["nwh"].eq(1),
})
meta["h"] = meta[JETS].sum(axis=1)
meta["jet_any"] = meta["h"].gt(0)

int8_cols = ["n_lines", "label", "marking", *JETS, "h", "jet_any"]
meta = meta.astype({c: "int8" for c in int8_cols}).reset_index()   # row order = CSV order; reuse for all matrices

assert meta["label"].sum() == N_ANOMALOUS_ENTRIES
meta.head()

,posting_id,fy,n_lines,label,marking,J1,J2,J3,J4,h,jet_any
0,a_1,2023,3,0,0,0,0,0,0,0,0
1,a_2,2023,3,0,0,0,0,0,0,0,0
2,a_3,2023,3,0,0,0,0,0,0,0,0
3,a_4,2023,3,0,0,0,0,0,0,0,0
4,a_5,2023,3,0,0,0,0,0,0,0,0


## 4. Validation: JETs re-implemented from raw fields (`src/jets.py`)
J1 keeps all ties at the 1,000th amount; the dataset `top_n` takes exactly 1,000 per year,
so J1 is expected to differ slightly. J2–J4 should match exactly.

In [6]:
# cell_01_06
raw = (apply_jets(build_entries(lines))
       .rename(columns={"in_jet": "jet_any"})[[*JETS, "jet_any"]]
       .astype("int8"))

m = meta.set_index("posting_id")
validation = pd.DataFrame([
    {
        "rule": r,
        "raw_pop": int(raw[r].sum()),
        "dataset_pop": int(m[r].sum()),
        "disagreements": int(raw[r].ne(m[r]).sum()),
        "anomalies_in_dataset_pop": int(m.loc[m[r] == 1, "label"].sum()),
        "precision_%": round(100 * m.loc[m[r] == 1, "label"].mean(), 2),
    }
    for r in [*JETS, "jet_any"]
]).set_index("rule")

assert validation.loc[["J2", "J3", "J4"], "disagreements"].eq(0).all()
validation

,raw_pop,dataset_pop,disagreements,anomalies_in_dataset_pop,precision_%
rule,,,,,
J1,2209,2000,209,196,9.80
J2,402,402,0,190,47.26
J3,8207,8207,0,310,3.78
J4,16493,16493,0,415,2.52
jet_any,24329,24183,146,798,3.30


## 5. Summary tables (main setting)

In [7]:
# cell_01_07
anoms = meta[meta["label"] == 1]
per_marking = anoms.groupby("marking")[[*JETS, "jet_any"]].sum()
per_marking.insert(0, "n", anoms.groupby("marking").size())
per_marking["recall_union_%"] = (100 * per_marking["jet_any"] / per_marking["n"]).round(1)

print(f"JET union recall: {anoms['jet_any'].mean():.4f}")
per_marking

JET union recall: 0.9388


,n,J1,J2,J3,J4,jet_any,recall_union_%
marking,,,,,,,
1,120,118,120,9,19,120,100.0
2,60,25,60,7,25,60,100.0
3,120,2,0,120,0,120,100.0
4,120,2,0,120,0,120,100.0
5,100,4,1,11,100,100,100.0
6,250,41,9,36,250,250,100.0
7,60,4,0,6,15,22,36.7
8,20,0,0,1,6,6,30.0


In [8]:
# cell_01_08
by_h = (meta[meta["h"] > 0]
        .groupby("h")["label"].agg(entries="size", anomalies="sum"))
by_h["precision_%"] = (100 * by_h["anomalies"] / by_h["entries"]).round(2)
by_h

,entries,anomalies,precision_%
h,,,
1,21387,543,2.54
2,2677,200,7.47
3,115,52,45.22
4,4,3,75.00


In [9]:
# cell_01_09
by_fy = (meta.assign(anoms_in_jet=meta["label"] & meta["jet_any"])
         .groupby("fy")
         .agg(entries=("posting_id", "size"), anomalies=("label", "sum"),
              jet_union=("jet_any", "sum"), anoms_in_jet=("anoms_in_jet", "sum")))
by_fy

,entries,anomalies,jet_union,anoms_in_jet
fy,,,,
2023,51064,423,12177,402
2024,51365,427,12006,396


## 6. Save
Parquet keeps dtypes and is readable from any Python/R version (needs `pyarrow`).

In [10]:
# cell_01_10
meta.to_parquet(OUT / "meta.parquet", index=False)
validation.to_csv(OUT / "jet_validation.csv")
print("saved", OUT / "meta.parquet", meta.shape)

saved C:\MyWorks\Thesis\Jet\data\derived\meta.parquet (102429, 11)
